# 09 — Giải thích và ba ca dự báo sai

Dự báo log return phiên kế tiếp, sau đó suy ra giá và hướng. Chạy các ô từ trên xuống. Dữ liệu dừng tại 30/09/2025.

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Permutation Importance là phân tích chính

Dùng XGBoost 32 biến đã học Train. Tráo một cột Test 10 lần, đo RMSE tăng bao nhiêu. Giá trị âm nghĩa là việc tráo làm điểm tốt hơn trên mẫu này; không phải tác động nhân quả.

In [ ]:
from explain import explain
summary=explain()
display(pd.read_csv(ROOT/'reports/permutation_importance.csv').head(15))
display(Image(filename=str(ROOT/'figures/permutation_importance.png')))

## Ba sai số giá lớn nhất của mô hình chọn bằng CV

VN-Index cùng phiên được xem sau đánh giá để mô tả bối cảnh. Mô hình không được dùng dữ liệu đó trước khi dự báo. Không suy diễn một tin tức là nguyên nhân khi chưa có bằng chứng.

In [ ]:
display(pd.read_csv(ROOT/'reports/error_cases.csv'))

## Phụ lục SHAP

SHAP phân chia dự báo thành đóng góp so với nền Train. Mô hình dùng 100 hàng Train làm nền và tối đa 150 hàng Test để giải thích. Không dùng biểu đồ Test để chỉnh lại mô hình hiện tại.

In [ ]:
display(Image(filename=str(ROOT/'figures/appendix_shap.png')))